### Implementing FunkSVD

In this notebook we will take a look at writing our own function that performs FunkSVD, which will follow the steps you saw in the previous video.  If you find that you aren't ready to tackle this task on your own, feel free to skip to the following video where you can watch as I walk through the steps.

To test our algorithm, we will run it on the subset of the data you worked with earlier.  Run the cell below to get started.

In [1]:
import numpy as np
import pandas as pd

# Read in the datasets
movies = pd.read_csv('data/movies_clean.csv')
reviews = pd.read_csv('data/reviews_clean.csv')

del movies['Unnamed: 0']
del reviews['Unnamed: 0']

# Create user-by-item matrix
user_items = reviews[['user_id', 'movie_id', 'rating', 'timestamp']]
user_by_movie = user_items.groupby(['user_id', 'movie_id'])['rating'].max().unstack()



# Create data subset
ratings_mat = np.matrix(user_by_movie)
print(ratings_mat)

[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]


`1.` You will use the **user_movie_subset** matrix to show that your FunkSVD algorithm will converge.  In the below cell, use the comments and document string to assist you as you complete writing your own function to complete FunkSVD.  You may also want to try to complete the funtion on your own without the assistance of comments.  You may feel free to remove and add to the function in any way that gets you a working solution!

**Notice:** There isn't a sigma matrix in this version of matrix factorization.

In [2]:
# A primeira célula é preservada; o recorte para o exercício é feito aqui.
subset_movie_ids = [75314, 68646, 99685]
user_movie_subset = user_by_movie[subset_movie_ids].dropna(axis=0)
ratings_mat = user_movie_subset.to_numpy(dtype=float)

def FunkSVD(
    ratings_mat, latent_features=4, learning_rate=0.0001, iters=100,
    random_state=42, return_history=False
):
    '''
    This function performs matrix factorization using a basic form of FunkSVD with no regularization

    INPUT:
    ratings_mat - (numpy array) a matrix with users as rows, movies as columns, and ratings as values
    latent_features - (int) the number of latent features used
    learning_rate - (float) the learning rate
    iters - (int) the number of iterations
    random_state - (int) seed used for reproducible initialization and shuffling
    return_history - (bool) whether to return the MSE from every iteration

    OUTPUT:
    user_mat - (numpy array) a user by latent feature matrix
    movie_mat - (numpy array) a latent feature by movie matrix
    history - (pandas dataframe, optional) iteration and MSE history
    '''

    # Normaliza a entrada e identifica suas dimensões.
    ratings = np.asarray(ratings_mat, dtype=float)
    if ratings.ndim != 2:
        raise ValueError('ratings_mat must be a two-dimensional matrix.')
    if latent_features < 1 or learning_rate <= 0 or iters < 1:
        raise ValueError('latent_features, learning_rate and iters must be positive.')

    n_users, n_movies = ratings.shape

    # NaN representa uma interação ausente e não participa do treinamento.
    observed_pairs = np.argwhere(~np.isnan(ratings))
    num_ratings = len(observed_pairs)
    if num_ratings == 0:
        raise ValueError('ratings_mat must contain at least one observed rating.')

    # Inicializa os fatores latentes com números pequenos e reproduzíveis.
    rng = np.random.default_rng(random_state)
    user_mat = rng.random((n_users, latent_features))
    movie_mat = rng.random((latent_features, n_movies))

    mse_history = []
    report_every = max(1, iters // 10)
    print('Optimization Statistics')
    print('Iteration | Mean Squared Error')

    # Cada época visita todas as avaliações conhecidas em uma ordem aleatória.
    for iteration in range(1, iters + 1):
        for pair_position in rng.permutation(num_ratings):
            user_idx, movie_idx = observed_pairs[pair_position]

            # A previsão é o produto dos fatores do usuário e do filme.
            prediction = np.dot(
                user_mat[user_idx], movie_mat[:, movie_idx]
            )
            error = ratings[user_idx, movie_idx] - prediction

            # Cópias permitem atualizar U e V com o mesmo gradiente calculado.
            user_features = user_mat[user_idx].copy()
            movie_features = movie_mat[:, movie_idx].copy()
            user_mat[user_idx] += (
                2 * learning_rate * error * movie_features
            )
            movie_mat[:, movie_idx] += (
                2 * learning_rate * error * user_features
            )

        # Mede o MSE usando apenas posições originalmente observadas.
        reconstructed = user_mat @ movie_mat
        observed_errors = (
            ratings[~np.isnan(ratings)]
            - reconstructed[~np.isnan(ratings)]
        )
        mse = float(np.mean(observed_errors ** 2))
        mse_history.append(mse)

        # Exibe checkpoints suficientes para observar a convergência sem poluir a saída.
        if iteration == 1 or iteration % report_every == 0 or iteration == iters:
            print(f'{iteration:9d} | {mse:.6f}')

    if return_history:
        history = pd.DataFrame({
            'iteration': np.arange(1, iters + 1),
            'mse': mse_history
        })
        return user_mat, movie_mat, history

    return user_mat, movie_mat

`2.` Try out your function on the **user_movie_subset** dataset.  First try 3 latent features, a learning rate of 0.005, and 10 iterations.  When you take the dot product of the resulting U and V matrices, how does the resulting **user_movie** matrix compare to the original subset of the data?

In [3]:
# Treina uma primeira versão curta para observar o início da convergência.
user_mat_10, movie_mat_10, history_10 = FunkSVD(
    ratings_mat, latent_features=3, learning_rate=0.005,
    iters=10, return_history=True
)

Optimization Statistics
Iteration | Mean Squared Error
        1 | 44.747838
        2 | 29.104907
        3 | 14.768204
        4 | 5.866238
        5 | 2.133260
        6 | 0.905045
        7 | 0.624493
        8 | 0.543009
        9 | 0.511225
       10 | 0.501853


In [4]:
# Reconstrói a matriz a partir dos dois fatores aprendidos.
predicted_10 = user_mat_10 @ movie_mat_10
predicted_10_df = pd.DataFrame(
    predicted_10, index=user_movie_subset.index,
    columns=user_movie_subset.columns
)

print('Predições após 10 épocas:')
display(predicted_10_df.round(2))
print('Avaliações reais:')
display(user_movie_subset)
mse_10 = float(history_10['mse'].iloc[-1])
print(f'MSE após 10 épocas: {mse_10:.6f}')

Predições após 10 épocas:


movie_id,75314,68646,99685
user_id,,,
2213,7.64,9.21,8.33
2223,7.15,8.48,7.68
2942,7.37,9.20,8.30
3298,8.44,10.20,9.24
3424,7.92,9.81,8.87
5205,7.54,9.43,8.52


Avaliações reais:


movie_id,75314,68646,99685
user_id,,,
2213,7.0,10.0,8.0
2223,6.0,10.0,7.0
2942,8.0,9.0,8.0
3298,8.0,10.0,10.0
3424,9.0,9.0,9.0
5205,8.0,9.0,9.0


MSE após 10 épocas: 0.501853


Após 10 épocas, as previsões já seguem a escala das avaliações reais, mas ainda há diferenças visíveis. O MSE caiu rapidamente, indicando que os fatores latentes estão aprendendo os padrões da matriz, embora o treinamento ainda não tenha convergido.

`3.` Let's try out the function again on the **user_movie_subset** dataset.  This time we will again use 3 latent features and a learning rate of 0.005.  However, let's bump up the number of iterations to 300.  When you take the dot product of the resulting U and V matrices, how does the resulting **user_movie** matrix compare to the original subset of the data?  What do you notice about the `mean squared error` at the end of each training iteration?

In [5]:
# Repete o treinamento por mais épocas, mantendo a mesma configuração e semente.
user_mat_300, movie_mat_300, history_300 = FunkSVD(
    ratings_mat, latent_features=3, learning_rate=0.005,
    iters=300, return_history=True
)

Optimization Statistics
Iteration | Mean Squared Error
        1 | 44.747838
       30 | 0.406960
       60 | 0.262264
       90 | 0.098291
      120 | 0.035801
      150 | 0.016371
      180 | 0.006739
      210 | 0.002393
      240 | 0.000776
      270 | 0.000253
      300 | 0.000089


In [6]:
# Compara novamente a reconstrução com as avaliações originais.
predicted_300 = user_mat_300 @ movie_mat_300
predicted_300_df = pd.DataFrame(
    predicted_300, index=user_movie_subset.index,
    columns=user_movie_subset.columns
)

print('Predições após 300 épocas:')
display(predicted_300_df.round(2))
print('Avaliações reais:')
display(user_movie_subset)
mse_300 = float(history_300['mse'].iloc[-1])
print(f'MSE após 300 épocas: {mse_300:.6f}')

Predições após 300 épocas:


movie_id,75314,68646,99685
user_id,,,
2213,7.00,10.0,8.00
2223,5.99,10.0,7.01
2942,8.01,9.0,8.00
3298,8.01,10.0,9.98
3424,8.98,9.0,9.02
5205,8.01,9.0,8.99


Avaliações reais:


movie_id,75314,68646,99685
user_id,,,
2213,7.0,10.0,8.0
2223,6.0,10.0,7.0
2942,8.0,9.0,8.0
3298,8.0,10.0,10.0
3424,9.0,9.0,9.0
5205,8.0,9.0,9.0


MSE após 300 épocas: 0.000089


Com 300 épocas, a matriz reconstruída fica muito mais próxima da original. O MSE diminui progressivamente e se aproxima de zero, mostrando a convergência do FunkSVD para este pequeno conjunto completo. Em dados reais, regularização e validação seriam necessárias para evitar sobreajuste.

In [7]:
# Resume quantitativamente o ganho obtido com o treinamento mais longo.
convergence_summary = pd.DataFrame({
    'epochs': [10, 300],
    'final_mse': [mse_10, mse_300]
})

# O exercício converge se aumentar o número de épocas reduzir o erro final.
assert mse_300 < mse_10
assert predicted_300.shape == ratings_mat.shape
convergence_summary

,epochs,final_mse
0,10,0.501853
1,300,0.000089
